# TSFM RUL benchmark — XJTU-SY bearings

**Run all** clones the code from GitHub, then runs the campaign for the XJTU-SY bearing
dataset. One of three parallel family notebooks (see `cmapss.ipynb`, `ncmapss.ipynb`).

## 1. Setup — clone the code, mount Drive for data

In [ ]:
# Install the non-default deps (Colab already ships torch/numpy/pandas/sklearn/matplotlib).
%pip install -q "chronos-forecasting>=2.0.0" "coral-pytorch==1.4.0" "lightgbm>=4.0" "sktime>=1.0" "numba>=0.59" "safetensors>=0.4" "h5py>=3.10"

import os, sys, subprocess, torch
from google.colab import drive

# 1. Mount Drive — it holds ONLY Data/, the embedding cache/, and results/ (what persists).
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Clone the CODE fresh from GitHub into Colab's ephemeral disk, so Drive never mirrors
#    the repo and you never re-upload changes. Re-run-safe: fast-forwards if already cloned.
REPO_URL    = 'https://github.com/blozanod/Predictive-Maintenance-LSTM.git'
REPO_BRANCH = 'main'                       # branch to run (a public repo needs no token)
CLONE_DIR   = '/content/Predictive-Maintenance-LSTM'
if not os.path.isdir(os.path.join(CLONE_DIR, '.git')):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH,
                    REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(['git', '-C', CLONE_DIR, 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', CLONE_DIR, 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', CLONE_DIR, 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)

# 3. Put the clone first on sys.path so `import src.*` resolves to the fresh code.
if CLONE_DIR in sys.path:
    sys.path.remove(CLONE_DIR)
sys.path.insert(0, CLONE_DIR)

print('repo :', CLONE_DIR, '@', REPO_BRANCH)
print('GPU  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Config

`src/config.py` is the single source of truth. The values below are the recorded FD001
ablation winner (CHANGES.md §12); the campaign runs every dataset at this config and
auto-names results/figures `<dataset>_<model>_…`. `sensor_columns` resolves per dataset
automatically (CHANGES.md §24).

**On Drive you now keep ONLY the notebooks and the data** — the code is cloned in the Setup
cell above. Drop the raw data under one `data_root`, one subfolder per family:

```
Data/
  XJTU-SY/        35Hz12kN/  37.5Hz11kN/  40Hz10kN/   (each holds BearingC_B/ folders)
                  # may instead be named XJTU-SY_Bearing_Datasets, or nested one level — both load (CHANGES.md §26)
```

Set `DRIVE` below to the Drive folder that holds your `Data/` (and where `cache/` +
`results/` will be written).

In [ ]:
from src.config import Config

# Point this at YOUR Drive folder holding Data/ (the only thing that must live on Drive).
DRIVE = '/content/drive/MyDrive/Predictive Maintenance LSTM'
config = Config(
    data_root=f'{DRIVE}/Data',
    cache_dir=f'{DRIVE}/cache',
    results_dir=f'{DRIVE}/results',
    # --- the recorded ablation winner (CHANGES.md §12); the campaign uses these ---
    tsfm_context_length=256,
    head_features='emb+locscale',
    pooling='mean',
    # experiment_name='v2',            # optional extra prefix on top of <dataset>_<model>
    # embed_batch_size=64,             # lower for a T4 (esp. N-CMAPSS: 37 channels)
)
config

## 3. Campaign — XJTU-SY bearings

`run_campaign(config, datasets=…)` is restricted to **XJTU-SY** so this notebook runs on
its own Colab runtime, in parallel with the other family notebooks. Per combo (restartable,
resumable): Stage A cache → sweep → fairness arms → horizon eval → figures. Missing datasets
are skipped with a notice. Per-dataset protocol choices come from
`campaign.DEFAULT_DATASET_OVERRIDES` (CHANGES.md §30).

In [ ]:
import torch
from src.campaign import run_campaign, DEFAULT_DATASET_OVERRIDES
from src.datasets import xjtu

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DEFAULT_DATASET_OVERRIDES   # inspect the recorded per-dataset protocol choices

# datasets=… restricts the run-all to THIS family; the list is derived from the registry so
# it self-maintains. dataset_overrides=None (default) => the recorded DEFAULT_DATASET_OVERRIDES.
summary = run_campaign(config, datasets=list(xjtu.DATASETS), device=device)
summary